# Train Conv-VAE-Neo MultiView Fusion

Initialize per-view mean encoders from a completed Neo VAE and train the fusion and proprioception heads.

In [ ]:
import pathlib
import pprint
import sys
sys.path.append("..")

import numpy as np
import torch
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"
from sensorprocessing.conv_vae_neo_multiview_fusion import (
    ConvVAENeoMultiViewFusionModel, train,
)
from sensorprocessing.multiview_data import make_multiview_dataloaders

## Exp/run parameters

In [ ]:
creation_style = "exist-ok"
expruns_path = None
results_path = None
epochs = None
experiment = "sensorprocessing_conv_vae_neo_multiview_fusion"
run = "sp_vae_neo_multiview_fusion_128_256px"

In [ ]:
if expruns_path:
    expruns_path = pathlib.Path(expruns_path)
    if not expruns_path.exists():
        raise FileNotFoundError(expruns_path)
    Config().set_exprun_path(expruns_path)
    Config().copy_experiment(experiment)
    Config().copy_experiment("sensorprocessing_conv_vae_neo")
    Config().copy_experiment("robot_al5d")
    Config().copy_experiment("demonstration")
if results_path:
    results_path = pathlib.Path(results_path)
    if not results_path.exists():
        raise FileNotFoundError(results_path)
    Config().set_results_path(results_path)

exp = Config().get_experiment(experiment, run, creation_style=creation_style)
pprint.pprint(exp)

## Train or resume

In [ ]:
exp.start_timer("training")
try:
    model = train(exp, epochs=epochs)
finally:
    exp.end_timer("training")

## Load the best model and evaluate proprioception

In [ ]:
device = Config().runtime["device"]
checkpoint_path = pathlib.Path(exp["data_dir"], "checkpoints", "best_model.pth")
# To load an intermediate checkpoint instead, replace checkpoint_path with:
# checkpoint_path = pathlib.Path(exp["data_dir"], "checkpoints", "epoch_000100.pth")
if not checkpoint_path.is_file():
    raise FileNotFoundError(checkpoint_path)
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model = ConvVAENeoMultiViewFusionModel(exp).to(device)
state = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
model.load_state_dict(state, strict=True)
model.eval()
print(f"Loaded checkpoint: {checkpoint_path}")

In [ ]:
robot_exp = Config().get_experiment(exp["robot_exp"], exp["robot_run"])
_, validation_loader = make_multiview_dataloaders(exp, robot_exp=robot_exp)
predictions, targets = [], []
with torch.no_grad():
    for batch_views, batch_targets in validation_loader:
        batch_views = [view.to(device) for view in batch_views]
        predictions.append(model(batch_views).cpu())
        targets.append(batch_targets)
predictions = torch.cat(predictions).numpy()
targets = torch.cat(targets).numpy()
field_names = ["height", "distance", "heading", "wrist_angle", "wrist_rotation", "gripper"]
for index, name in enumerate(field_names[:predictions.shape[1]]):
    rmse = np.sqrt(np.mean((predictions[:, index] - targets[:, index]) ** 2))
    mae = np.mean(np.abs(predictions[:, index] - targets[:, index]))
    print(f"{name:<16} RMSE {rmse:.4f}   MAE {mae:.4f}")
print(f"Overall RMSE {np.sqrt(np.mean((predictions - targets) ** 2)):.4f}")

sample_views, _ = next(iter(validation_loader))
sample_views = [view.to(device) for view in sample_views]
with torch.no_grad():
    features = model.extract_features(sample_views)
    scores = model.fusion.view_scores(features)
if scores is not None:
    print("Mean view weights:", scores.mean(dim=0).cpu().numpy())